# Demo PySpark — Dari DataFrame sampai Data Warehouse

Notebook ini adalah materi pengenalan **Apache Spark (PySpark 4)** dengan dataset penjualan toko online fiktif:

| File | Isi |
|---|---|
| `/workspace/data/demo_sales.csv` | 301 transaksi (ada data kotor: null, duplikat, huruf besar tidak konsisten) |
| `/workspace/data/demo_customers.csv` | 50 customer |

**Daftar isi**
1. SparkSession
2. Membuat DataFrame
3. RDD (Resilient Distributed Dataset)
4. Membaca CSV, JSON & schema
5. Transformasi dasar (select, filter, withColumn)
6. Lazy evaluation
7. Data cleaning
8. Agregasi
9. Join
10. Spark SQL
11. Window function
12. Menyimpan hasil (Parquet/CSV)
13. Load ke PostgreSQL di laptop (JDBC) — untuk GC6

## 1. SparkSession
`SparkSession` adalah pintu masuk ke semua fitur Spark. Di container ini Spark berjalan mode **local** (`local[2]` = 2 core, driver memory 1 GB — diatur di `spark-defaults.conf`).

Selama session aktif, **Spark UI** bisa dibuka di http://localhost:4040 untuk melihat job, stage, dan task.

In [ ]:
# Import library dalam satu cell
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType
from pyspark.sql.window import Window

# Membuat (atau mengambil) SparkSession
spark = SparkSession.builder.appName("demo-spark").getOrCreate()

print("Versi Spark :", spark.version)
print("Master      :", spark.sparkContext.master)

## 2. Membuat DataFrame dari data Python
DataFrame Spark mirip tabel: punya kolom bernama dan tipe data. Bedanya dengan pandas, data Spark **terdistribusi** ke banyak partisi.

In [ ]:
data = [("Alice", 34, "Jakarta"), ("Bob", 45, "Bandung"), ("Cathy", 29, "Surabaya")]
df_people = spark.createDataFrame(data, ["Name", "Age", "City"])

df_people.show()
df_people.printSchema()

## 3. RDD (Resilient Distributed Dataset)
RDD adalah abstraksi dasar Spark: kumpulan data **immutable** yang dipecah ke beberapa **partisi** dan diproses paralel.

| Sifat | Arti |
|---|---|
| In-memory | data diproses di memori → cepat |
| Immutable | RDD tidak diubah; setiap transformasi menghasilkan RDD baru |
| Fault tolerant | partisi yang hilang bisa dihitung ulang dari *lineage* |
| Lazy | transformasi baru dijalankan saat ada action |
| Partitioning | data dibagi ke partisi untuk paralelisme |

Saat ini kita lebih sering memakai **DataFrame** (lebih cepat karena dioptimasi Catalyst), tetapi DataFrame dibangun di atas RDD.

In [ ]:
sc = spark.sparkContext

rdd = sc.parallelize(range(1, 11), numSlices=2)          # RDD dari list, 2 partisi
print("Partisi      :", rdd.getNumPartitions())
print("Isi partisi  :", rdd.glom().collect())              # lihat isi tiap partisi

squared_even = rdd.map(lambda x: x * x).filter(lambda x: x % 2 == 0)   # transformation (lazy)
print("Kuadrat genap:", squared_even.collect())                       # action
print("Total        :", squared_even.reduce(lambda a, b: a + b))

In [ ]:
# Word count klasik dengan RDD
lines = sc.parallelize(["spark itu cepat", "airflow untuk orkestrasi", "spark dan airflow"])
word_count = lines.flatMap(lambda l: l.split()).map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)
print(sorted(word_count.collect(), key=lambda x: -x[1]))

# RDD -> DataFrame dengan createDataFrame()
df_wc = spark.createDataFrame(word_count, ["word", "count"])
df_wc.orderBy("count", ascending=False).show()

## 4. Membaca CSV, JSON & schema
Ada dua cara menentukan tipe kolom:
- `inferSchema=True` → Spark menebak tipe (praktis, tapi membaca data 2x)
- **schema eksplisit** → lebih cepat dan pasti, disarankan untuk pipeline produksi

In [ ]:
# Cara 1: inferSchema
df_infer = spark.read.csv("/workspace/data/demo_sales.csv", header=True, inferSchema=True)
df_infer.printSchema()

In [ ]:
# Cara 2: schema eksplisit
sales_schema = StructType([
    StructField("OrderID", StringType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerID", StringType()),
    StructField("Product", StringType()),
    StructField("Category", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", LongType()),
    StructField("PaymentMethod", StringType()),
])
sales = spark.read.csv("/workspace/data/demo_sales.csv", header=True, schema=sales_schema)

sales.show(5)
print("Jumlah baris :", sales.count())
print("Jumlah partisi:", sales.rdd.getNumPartitions())

In [ ]:
# Statistik deskriptif kolom numerik
sales.describe("Quantity", "UnitPrice").show()

In [ ]:
# Sumber data lain: JSON & TXT
# Tulis contoh JSON dulu (satu objek per baris = format JSON Lines yang dibaca Spark)
import json, os
os.makedirs("/workspace/data/output", exist_ok=True)
with open("/workspace/data/output/demo_products.json", "w") as f:
    for p in [{"Product": "Laptop", "Brand": "Lenovo", "Warranty": 2}, {"Product": "Mouse", "Brand": "Logitech", "Warranty": 1}]:
        f.write(json.dumps(p) + "\n")

spark.read.json("/workspace/data/output/demo_products.json").show()

# File teks: setiap baris jadi satu row dengan kolom 'value'
spark.read.text("/workspace/data/demo_customers.csv").show(3, truncate=False)

## 5. Transformasi dasar
| Method | Fungsi |
|---|---|
| `select` | memilih kolom |
| `filter` / `where` | memfilter baris |
| `withColumn` | menambah/mengubah kolom |
| `withColumnRenamed` | mengganti nama kolom |
| `F.when(...).otherwise(...)` | if-else |

In [ ]:
sales_tx = (
    sales
    .withColumn("TotalPrice", F.col("Quantity") * F.col("UnitPrice"))         # kolom baru
    .withColumn("OrderMonth", F.date_format("OrderDate", "yyyy-MM"))           # format tanggal
    .withColumn("PriceTier", F.when(F.col("UnitPrice") >= 1_000_000, "High")   # if-else
                               .when(F.col("UnitPrice") >= 100_000, "Medium")
                               .otherwise("Low"))
)

sales_tx.select("OrderID", "Product", "Quantity", "UnitPrice", "TotalPrice", "PriceTier") \
        .filter(F.col("Category") == "Electronics") \
        .show(5)

## 6. Lazy evaluation
Transformasi (`select`, `filter`, `withColumn`, ...) **tidak langsung dijalankan** — Spark hanya mencatat rencana (*logical plan*).
Eksekusi baru terjadi saat ada **action** (`show`, `count`, `collect`, `write`, ...).

`explain()` menampilkan rencana eksekusi yang sudah dioptimasi oleh Catalyst Optimizer.

In [ ]:
plan = sales_tx.filter(F.col("TotalPrice") > 1_000_000).select("OrderID", "TotalPrice")
plan.explain()   # belum ada data yang diproses
plan.count()     # action -> job dijalankan (lihat di Spark UI)

## 7. Data cleaning
Langkah umum:
1. Cek & hapus duplikat
2. Cek missing value per kolom
3. Isi / hapus missing value
4. Standarisasi teks

In [ ]:
# 6.1 Duplikat berdasarkan OrderID
sales.groupBy("OrderID").count().filter("count > 1").show()

In [ ]:
# 6.2 Jumlah null per kolom
sales.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sales.columns]).show()

In [ ]:
# 6.3 Inkonsistensi huruf besar pada Product
sales.groupBy("Product").count().orderBy("Product").show()

In [ ]:
sales_clean = (
    sales
    .dropDuplicates(["OrderID"])                                # hapus duplikat
    .withColumn("Product", F.initcap(F.trim("Product")))        # LAPTOP -> Laptop
    .fillna({"PaymentMethod": "Unknown"})                       # isi kategori kosong
    .dropna(subset=["Quantity"])                                # hapus baris tanpa Quantity
    .withColumn("TotalPrice", F.col("Quantity") * F.col("UnitPrice"))
)

print("Sebelum:", sales.count(), "| Sesudah:", sales_clean.count())
sales_clean.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sales_clean.columns]).show()

## 8. Agregasi
`groupBy` + `agg` untuk ringkasan per kelompok.

In [ ]:
(
    sales_clean.groupBy("Category")
    .agg(
        F.count("OrderID").alias("TotalOrders"),
        F.sum("Quantity").alias("TotalQty"),
        F.sum("TotalPrice").alias("Revenue"),
        F.round(F.avg("TotalPrice"), 0).alias("AvgOrderValue"),
    )
    .orderBy(F.desc("Revenue"))
    .show()
)

## 9. Join
Gabungkan transaksi dengan data customer untuk analisis per kota.
Jenis join: `inner`, `left`, `right`, `outer`, `left_anti` (yang tidak punya pasangan), dll.

In [ ]:
customers = spark.read.csv("/workspace/data/demo_customers.csv", header=True, inferSchema=True)

sales_cust = sales_clean.join(customers, on="CustomerID", how="left")

(
    sales_cust.groupBy("City")
    .agg(F.countDistinct("CustomerID").alias("Customers"), F.sum("TotalPrice").alias("Revenue"))
    .orderBy(F.desc("Revenue"))
    .show()
)

## 10. Spark SQL
DataFrame bisa didaftarkan sebagai *temporary view* lalu di-query dengan SQL biasa. Hasilnya tetap DataFrame.

In [ ]:
sales_cust.createOrReplaceTempView("sales")

spark.sql('''
    SELECT Segment,
           PaymentMethod,
           COUNT(*)        AS Orders,
           SUM(TotalPrice) AS Revenue
    FROM sales
    GROUP BY Segment, PaymentMethod
    ORDER BY Segment, Revenue DESC
''').show()

## 11. Window function
Menghitung nilai per baris **dengan melihat baris lain di kelompok yang sama** tanpa menggabungkan baris (berbeda dengan `groupBy`).
Contoh: ranking produk di tiap kategori dan running total revenue bulanan.

> Window tanpa `partitionBy` (contoh running total) memunculkan warning *No Partition Defined for Window operation* karena semua data dipindah ke 1 partisi. Aman untuk data kecil/hasil agregasi, tapi hindari untuk data besar.

In [ ]:
product_rev = sales_clean.groupBy("Category", "Product").agg(F.sum("TotalPrice").alias("Revenue"))

w_rank = Window.partitionBy("Category").orderBy(F.desc("Revenue"))
product_rev.withColumn("RankInCategory", F.dense_rank().over(w_rank)).orderBy("Category", "RankInCategory").show()

In [ ]:
monthly = (
    sales_clean.withColumn("OrderMonth", F.date_format("OrderDate", "yyyy-MM"))
    .groupBy("OrderMonth").agg(F.sum("TotalPrice").alias("Revenue"))
)

w_running = Window.orderBy("OrderMonth").rowsBetween(Window.unboundedPreceding, Window.currentRow)
monthly.withColumn("RunningRevenue", F.sum("Revenue").over(w_running)).orderBy("OrderMonth").show(12)

## 12. Menyimpan hasil
- **Parquet** → format kolumnar, menyimpan schema, jauh lebih cepat & kecil dari CSV. Format favorit data lake.
- `partitionBy` → memecah folder output per nilai kolom (mempercepat filter).
- Output Spark berupa **folder** berisi beberapa file `part-*`.

In [ ]:
output_path = "/workspace/data/output/sales_clean_parquet"
sales_clean.write.mode("overwrite").partitionBy("Category").parquet(output_path)

# Baca kembali hanya partisi Electronics
spark.read.parquet(output_path).filter(F.col("Category") == "Electronics").show(3)

## 13. Load ke PostgreSQL di laptop (JDBC) — untuk GC6
Tujuan bagian ini: menyimpan DataFrame Spark ke **PostgreSQL yang terpasang di laptop** (yang dibuka lewat pgAdmin).

**Syarat:** PostgreSQL di laptop sedang berjalan, database `dwh` sudah dibuat di pgAdmin, dan file `.env` sudah berisi password-nya (`PG_PASSWORD`, sama dengan password pgAdmin).

Notebook ini berjalan **di dalam container Docker**, jadi alamatnya berbeda dengan di pgAdmin:

| | Host | Port |
|---|---|---|
| pgAdmin / DBeaver (di laptop) | `localhost` | `5432` |
| Notebook ini (di dalam container) | **`host.docker.internal`** | `5432` |

`localhost` di dalam container artinya container itu sendiri, bukan laptop. Karena itu dipakai `host.docker.internal` (= "laptop saya").

Driver JDBC PostgreSQL sudah terpasang di image, jadi tidak perlu `spark.jars.packages`.

### 13.1 Ambil konfigurasi dari `.env`
Host, user, dan password **tidak ditulis di notebook**. Semuanya diambil dari file `.env`, sehingga notebook aman di-push ke GitHub:

```
PG_HOST=host.docker.internal
PG_PORT=5432
PG_DB=dwh
PG_USER=postgres
PG_PASSWORD=isi-password-postgres-kamu
```

Setelah mengubah `.env`, jalankan `docker compose up -d` di terminal supaya setting baru terbaca, lalu refresh JupyterLab.

In [ ]:
import os

belum_diisi = [k for k in ("PG_HOST", "PG_PORT", "PG_DB", "PG_USER", "PG_PASSWORD") if not os.getenv(k)]
if belum_diisi:
    raise SystemExit(f"Belum diisi di file .env: {belum_diisi}. Salin .env.example menjadi .env, isi, lalu jalankan docker compose up -d")

PG_HOST = os.environ["PG_HOST"]
PG_PORT = os.environ["PG_PORT"]
PG_DB = os.environ["PG_DB"]
PG_USER = os.environ["PG_USER"]
PG_PASSWORD = os.environ["PG_PASSWORD"]

jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
jdbc_props = {"user": PG_USER, "password": PG_PASSWORD, "driver": "org.postgresql.Driver"}
print("JDBC URL:", jdbc_url)

### 13.2 Tes koneksi
Cell ini mengecek koneksi dengan `psycopg2` dan memberi pesan yang jelas kalau gagal
(database belum dibuat, password salah, atau PostgreSQL belum jalan).

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB,
                            user=PG_USER, password=PG_PASSWORD, connect_timeout=5)
except psycopg2.OperationalError as e:
    msg = str(e)
    if "does not exist" in msg:
        raise SystemExit(f"Database {PG_DB} belum ada. Buat dulu di pgAdmin: klik kanan Databases > Create > Database > {PG_DB}")
    elif "password authentication failed" in msg:
        raise SystemExit("Password salah. Samakan PG_USER / PG_PASSWORD di file .env dengan yang dipakai di pgAdmin.")
    elif "Connection refused" in msg or "timeout" in msg:
        raise SystemExit("PostgreSQL di laptop tidak bisa dihubungi. Pastikan service PostgreSQL berjalan (cek pgAdmin bisa connect).")
    else:
        raise

with conn.cursor() as cur:
    cur.execute("SELECT version()")
    print("Terhubung ke:", cur.fetchone()[0].split(",")[0])
conn.close()

### 13.3 Tulis DataFrame ke PostgreSQL
`df.write.jdbc(url, "nama_tabel", mode, properties)`:

| mode | Arti |
|---|---|
| `overwrite` | hapus tabel lalu buat ulang dari DataFrame |
| `append` | tambah baris ke tabel yang sudah ada (tabel dibuat otomatis jika belum ada) |
| `ignore` | lewati jika tabel sudah ada |
| `error` (default) | gagal jika tabel sudah ada |

> **Untuk GC6:** kalau tabel fact/dimension sudah dibuat dengan DDL (file `.sql`, lengkap dengan primary key), pakai `mode="append"`.
> `overwrite` akan menghapus tabel itu beserta primary key / foreign key-nya. Nama dan tipe kolom DataFrame harus sesuai dengan tabelnya.

In [ ]:
# Contoh sederhana: tabel dimensi customer & tabel fakta penjualan
customers.write.jdbc(jdbc_url, "demo_dim_customer", mode="overwrite", properties=jdbc_props)

sales_clean.select("OrderID", "OrderDate", "CustomerID", "Product", "Quantity", "UnitPrice", "TotalPrice") \
           .write.jdbc(jdbc_url, "demo_fact_sales", mode="overwrite", properties=jdbc_props)

print("Tersimpan ke", jdbc_url)

### 13.4 Cek hasilnya
Baca balik dari PostgreSQL dengan query (query dibungkus kurung dan diberi alias), lalu cek juga di pgAdmin:
database `dwh` → *Schemas → public → Tables* → klik kanan *Refresh* → `demo_dim_customer`, `demo_fact_sales`.

In [ ]:
query = "(SELECT COUNT(*) AS jumlah_baris FROM demo_fact_sales) AS t"
spark.read.jdbc(jdbc_url, query, properties=jdbc_props).show()

### 13.5 (Opsional) Membaca tabel besar secara paralel
Untuk tabel besar, gunakan `column` (kolom numerik), `lowerBound`, `upperBound`, dan `numPartitions`
agar Spark membuka beberapa koneksi sekaligus dan membaca per rentang nilai.

In [ ]:
query_big = "(SELECT *, EXTRACT(MONTH FROM \"OrderDate\")::int AS order_month FROM demo_fact_sales) AS t"
df_parallel = spark.read.jdbc(
    url=jdbc_url,
    table=query_big,
    column="order_month",   # kolom pembagi partisi
    lowerBound=1,
    upperBound=12,
    numPartitions=4,
    properties=jdbc_props,
)
print("Partisi:", df_parallel.rdd.getNumPartitions(), "| Baris:", df_parallel.count())

## 14. Spark → pandas & menutup session
`toPandas()` membawa **seluruh data ke memori driver** — gunakan hanya untuk data yang sudah kecil (hasil agregasi), misalnya untuk visualisasi.

In [ ]:
summary_pdf = sales_clean.groupBy("Category").agg(F.sum("TotalPrice").alias("Revenue")).toPandas()
summary_pdf

In [ ]:
# Selalu hentikan session agar memori laptop dilepas
spark.stop()